# Layer 3 — Model Development: LoRA Fine-Tuning, Artifact Logging & Registry Promotion

**Architectural Layer:** Model Development  
**Toolchain:** Weights & Biases · Hugging Face Transformers · PEFT (LoRA/QLoRA) · MLflow  
**Objective:** Fine-tune a foundation model with parameter-efficient adapters, track experiments end-to-end, and promote artifacts through a model registry.

---

## 🧠 What is Model Development in MLOps and Why Does It Matter?

### The Old Way (Pre-2023)
Data scientists trained models in Jupyter notebooks, saved the weights to a local folder, emailed the file to an engineer, and hoped it worked in production. No tracking, no versioning, no audit trail.

### The Modern Way (2026)
Every experiment is tracked automatically. Every model artifact is versioned and registered. Every hyperparameter choice is logged. You can reproduce ANY previous experiment with a single command.

### 🤔 Why Can't We Just Train a Model and Deploy It?

| Problem | What Happens |
|---------|-------------|
| No experiment tracking | "Which model was better? I don't remember the settings I used" |
| No artifact versioning | "We deployed model_v2 but we need to rollback — where is model_v1?" |
| No registry | "Is this model approved for production? Who tested it?" |
| No reproducibility | "The model worked on my laptop but not in production" |

---

## 📚 Key Concepts Explained for Beginners

### What is Fine-Tuning?

**Analogy:** A foundation model (like GPT or Phi-2) is like a university graduate. They have general knowledge about many topics. **Fine-tuning** is like giving them specialized on-the-job training. They keep their general knowledge but become experts in YOUR specific task.

### 🤔 Why Fine-Tune Instead of Training from Scratch?

| Approach | Cost | Time | Data Needed | Result |
|----------|------|------|-------------|--------|
| **Train from scratch** | $100K-$10M | Weeks-months | Billions of tokens | General model |
| **Full fine-tuning** | $1K-$10K | Hours-days | Millions of tokens | Specialized model |
| **LoRA fine-tuning** | $10-$100 | Minutes-hours | Thousands of examples | Specialized model |

LoRA is 100-1000x cheaper than full fine-tuning with nearly the same quality. This is why it's the default approach in 2026.

### What is LoRA (Low-Rank Adaptation)?

**The problem:** A model like Phi-2 has 2.7 BILLION parameters. Fine-tuning ALL of them requires massive GPU memory and risks "catastrophic forgetting" (the model forgets its general knowledge).

**LoRA's solution:** Instead of modifying all 2.7B parameters, LoRA adds tiny "adapter" modules (< 1% of total parameters) that learn your task-specific behavior. The original parameters stay frozen.

```
Original model:  [2.7B parameters] ← FROZEN (unchanged)
LoRA adapters:   [~14M parameters] ← TRAINED (learns your task)
Combined output: General knowledge + Your specific task
```

### What is QLoRA (Quantized LoRA)?

QLoRA goes one step further: it **compresses** the frozen model from 16-bit to 4-bit precision, reducing memory usage by 4x.

| Method | Memory for 7B Model | Quality |
|--------|--------------------|---------|
| Full fine-tuning (16-bit) | ~28 GB GPU RAM | Best |
| LoRA (16-bit base) | ~14 GB GPU RAM | 99% of full |
| QLoRA (4-bit base) | ~4 GB GPU RAM | 97-99% of full |

**🤔 When to use QLoRA vs LoRA?**
- **QLoRA**: When GPU memory is limited (consumer GPUs, cloud budget constraints)
- **LoRA**: When you have enough GPU memory and want maximum quality
- **Full fine-tuning**: Only when you have massive compute budget AND LoRA quality isn't sufficient

### What is a Model Registry?

A **model registry** is like a library catalog for your trained models. It tracks:
- Which version of the model exists
- Who trained it and when
- What metrics it achieved
- What stage it's in: `Development` → `Staging` → `Production`

**Real-world scenario:**  
Model v3 is in Production. A data scientist trains v4 (Development). The QA team tests v4 (Staging). If it passes all tests, it's promoted to Production and v3 is archived.

### ⚖️ Trade-offs: W&B vs MLflow vs Comet

| Feature | Weights & Biases | MLflow | Comet |
|---------|-----------------|--------|-------|
| **Experiment tracking** | ⭐⭐⭐ Beautiful UI | ⭐⭐ Functional UI | ⭐⭐⭐ Good UI |
| **Model registry** | Basic | ⭐⭐⭐ Full lifecycle | ⭐⭐ Good |
| **Self-hosted** | Limited | ✅ Fully | Limited |
| **Cost** | Free tier + paid | Free + open-source | Free tier + paid |
| **Best for** | Experiment comparison | Full MLOps, registry | Experiment tracking |

**🤔 Why do we use BOTH W&B and MLflow?**  
W&B excels at experiment tracking (comparing runs, visualizing loss curves). MLflow excels at model registry (lifecycle management, deployment). Many teams use both.

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Training a foundation model from scratch takes millions of dollars. Seniors use PEFT (Parameter-Efficient Fine-Tuning) and LoRA to adapt giant open-source models to niche domains using a single GPU.

**Mental Model:** Full fine-tuning is rewriting an entire textbook. LoRA (Low-Rank Adaptation) is adding a tiny 10-page cheat sheet (adapters) to the back of the book, keeping the original book frozen.

**Common Junior Pitfall:** Failing to track experiments. Juniors tweak learning rates randomly; seniors use MLflow/W&B to rigorously track every hyperparameter, dataset version, and resulting metric.

---


## 1 · Experiment Tracking Setup

In [ ]:
# Cell 1 — Install dependencies
# transformers: Hugging Face's library for pretrained models
# accelerate: handles multi-GPU/mixed-precision training
# peft: Parameter-Efficient Fine-Tuning (LoRA, QLoRA)
# bitsandbytes: enables 4-bit/8-bit quantization
# trl: Transformer Reinforcement Learning (SFT trainer)
# wandb: Weights & Biases experiment tracking
# mlflow: model registry and artifact management
!pip install -q transformers accelerate peft bitsandbytes datasets wandb mlflow trl torch

In [ ]:
# Cell 2 — Configure Weights & Biases experiment tracking
#
# WHAT IS HAPPENING?
# We initialize a W&B "run" — a tracked experiment. Everything we log
# (metrics, configs, artifacts) will appear in the W&B web dashboard.
#
# WHY log the config upfront?
# So that when you compare 50 experiments later, you can filter by:
# "Show me all runs with learning_rate < 0.001 and lora_r >= 16"
#
# 🤔 What if I forget to log something?
# You'll have to re-run the experiment. This is why we log EVERYTHING
# upfront — it's much cheaper than re-training.

import wandb
import os

os.environ["WANDB_PROJECT"] = "mlops-2026-lora-finetuning"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # auto-save model checkpoints to W&B

run = wandb.init(
    project="mlops-2026-lora-finetuning",
    config={
        # Model selection
        "model_name": "microsoft/phi-2",  # 2.7B param model (fits on consumer GPU)
        # LoRA hyperparameters
        "lora_r": 16,            # rank: higher = more capacity but more memory
        "lora_alpha": 32,        # scaling factor: usually 2x the rank
        "lora_dropout": 0.05,    # dropout to prevent overfitting
        "target_modules": ["q_proj", "k_proj", "v_proj", "dense"],  # which layers get adapters
        # Quantization
        "quantization": "4bit",  # QLoRA: compress base model to 4-bit
        # Training hyperparameters
        "learning_rate": 2e-4,   # how fast the model learns
        "epochs": 3,             # how many times to see the full dataset
        "batch_size": 4,         # samples per GPU per step
        "gradient_accumulation_steps": 8,  # simulate batch_size of 32
        "max_seq_length": 1024,  # maximum context window
    },
    tags=["lora", "qlora", "peft", "instruction-tuning"],
)
config = wandb.config
print(f"✅ W&B run initialized: {run.url}")

### 🤔 Understanding LoRA Hyperparameters

| Parameter | What It Controls | Low Value | High Value |
|-----------|-----------------|-----------|------------|
| `lora_r` (rank) | Adapter capacity | Less learning, less memory | More learning, more memory |
| `lora_alpha` | Scaling factor | Less influence of adapters | More influence of adapters |
| `lora_dropout` | Regularization | Risk of overfitting | May underfit |
| `target_modules` | Which layers to adapt | Fewer adaptations | More flexibility |

**Rule of thumb:** Start with `r=16, alpha=32` and adjust based on results.

**🤔 What if `lora_r` is too low?**  
The adapters don't have enough capacity to learn your task. The model barely changes.

**🤔 What if `lora_r` is too high?**  
More memory usage, longer training, risk of overfitting. Diminishing returns above r=64.

In [ ]:
# Cell 3 — Configure MLflow for model registry
#
# MLflow serves a different purpose than W&B:
# - W&B: experiment tracking (compare runs, visualize metrics)
# - MLflow: model registry (version models, manage lifecycle stages)
#
# Together they cover the full development workflow.
#
# 🤔 Why SQLite for the tracking URI?
# For local development/demo. In production you'd use a proper server:
# mlflow.set_tracking_uri("http://mlflow-server.company.com:5000")

import mlflow

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("lora-finetuning-experiments")

print(f"✅ MLflow configured:")
print(f"   Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"   Experiment: lora-finetuning-experiments")

---

## 2 · Model Initialization with QLoRA

### 🤔 Understanding 4-bit Quantization

**What does "4-bit" mean?**  
Each model parameter is normally stored as a 16-bit or 32-bit floating point number. 4-bit quantization compresses each parameter into just 4 bits — that's 4-8x less memory.

**Analogy:** It's like compressing a photo from RAW (50MB) to JPEG (5MB). You lose some quality, but it's good enough for most purposes.

| Precision | Bits per Parameter | 2.7B Model Size | Quality |
|-----------|-------------------|-----------------|--------|
| FP32 | 32 bits | ~10.8 GB | Perfect |
| FP16/BF16 | 16 bits | ~5.4 GB | Near-perfect |
| INT8 | 8 bits | ~2.7 GB | Very good |
| NF4 (4-bit) | 4 bits | ~1.35 GB | Good (with QLoRA) |

**NF4 (NormalFloat4)** is a special 4-bit format designed specifically for neural network weights. It's better than naive 4-bit quantization because it accounts for the bell-curve distribution of typical model weights.

**🤔 When NOT to quantize?**
- When you need maximum quality (e.g., medical diagnosis models)
- When you have abundant GPU memory
- When you're doing full fine-tuning (not LoRA)

In [ ]:
# Cell 4 — Load base model with 4-bit quantization
#
# WHAT IS HAPPENING?
# 1. Configure 4-bit quantization (NF4 format, double quantization)
# 2. Load the tokenizer (converts text ↔ numbers)
# 3. Load the model with quantization applied
#
# IMPORTANT DETAILS:
# - bnb_4bit_compute_dtype=bfloat16: compute in 16-bit even though storage is 4-bit
#   This gives better training stability while keeping memory low.
# - bnb_4bit_use_double_quant=True: quantize the quantization constants too!
#   Saves an extra ~0.4 GB on a 7B model.
# - device_map="auto": automatically distribute model across available GPUs
# - use_cache=False: required for gradient checkpointing during training
#
# 🤔 Why set pad_token = eos_token?
# Many models (including Phi-2) don't define a padding token by default.
# Without one, batched training fails because sequences of different lengths
# can't be packed into a single tensor. We use the end-of-sequence token
# as padding — a common workaround.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = config.model_name

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Total parameters: {total_params/1e9:.2f}B")
print(f"   Quantization: 4-bit NF4 with double quantization")

In [ ]:
# Cell 5 — Inject LoRA adapters
#
# WHAT IS HAPPENING?
# 1. prepare_model_for_kbit_training: prepares the quantized model for LoRA
#    (enables gradient computation on frozen layers, fixes numerical issues)
# 2. LoraConfig: defines HOW the adapters are structured
# 3. get_peft_model: wraps the base model with LoRA adapters
#
# AFTER THIS:
# - Only ~0.5% of parameters are trainable (the adapters)
# - The other 99.5% are frozen (the original model)
# - Training only updates the adapter weights
#
# 🤔 What are "target_modules"?
# These specify WHICH layers in the transformer get LoRA adapters.
# q_proj, k_proj, v_proj = the Query, Key, Value projections in attention
# dense = the feed-forward layers
# More modules = more capacity but more memory and training time.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=config.lora_r,                      # rank of the adapter matrices
    lora_alpha=config.lora_alpha,          # scaling factor
    target_modules=config.target_modules,  # which layers to adapt
    lora_dropout=config.lora_dropout,      # regularization
    bias="none",                           # don't train bias terms
    task_type=TaskType.CAUSAL_LM,          # tell PEFT this is a language model
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / all_params

print(f"✅ LoRA adapters injected:")
print(f"   Trainable parameters: {trainable_params/1e6:.2f}M ({trainable_pct:.2f}%)")
print(f"   Frozen parameters:    {(all_params - trainable_params)/1e9:.2f}B")

wandb.log({"trainable_params_M": trainable_params / 1e6, "trainable_pct": trainable_pct})

### 📝 About Instruction-Tuning Data Format

**🤔 What format does the model expect?**

For instruction-following, we use a structured template:

```
### Instruction:
Explain gradient descent in simple terms.

### Response:
Gradient descent is an optimization algorithm that...<EOS>
```

The `<EOS>` (end-of-sequence) token tells the model when to stop generating.

**🤔 Why does the format matter so much?**  
The model learns to recognize the pattern: "When I see `### Instruction:`, I should read the question. When I see `### Response:`, I should generate an answer." If you change the format between training and serving, the model gets confused.

**Real-world scenario:**  
A company fine-tuned their model with `[USER]...[ASSISTANT]` format. At serving time, they accidentally used `Question:...Answer:` format. The model produced gibberish because it had never seen that pattern.

In [ ]:
# Cell 6 — Prepare instruction-tuning dataset
#
# We create training examples in the instruction-response format.
# In production, this data would come from your Feature Store (Notebook 02)
# or from curated datasets on Hugging Face.
#
# 🤔 Why repeat the data 200x?
# With only 5 unique examples, the model needs to see them many times.
# In production, you'd have thousands of UNIQUE examples.
# This is just for demonstration purposes.

from datasets import Dataset

raw_data = [
    {"instruction": "Explain gradient descent in simple terms.",
     "response": "Gradient descent is an optimization algorithm that iteratively adjusts parameters by moving in the direction of steepest decrease of a loss function."},
    {"instruction": "What is the difference between L1 and L2 regularization?",
     "response": "L1 regularization adds the absolute value of weights as a penalty, promoting sparsity. L2 adds the squared weights, promoting smaller but non-zero values."},
    {"instruction": "Describe the transformer attention mechanism.",
     "response": "The attention mechanism computes weighted sums of value vectors, where weights are determined by the compatibility between query and key vectors using scaled dot-product."},
    {"instruction": "What is batch normalization?",
     "response": "Batch normalization normalizes the inputs of each layer to have zero mean and unit variance across the mini-batch, stabilizing and accelerating training."},
    {"instruction": "Explain the bias-variance tradeoff.",
     "response": "The bias-variance tradeoff states that models with high bias underfit while models with high variance overfit. The goal is to find the sweet spot."},
] * 200

def format_instruction(sample: dict) -> dict:
    text = f"""### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['response']}{tokenizer.eos_token}"""
    return {"text": text}

dataset = Dataset.from_list(raw_data).map(format_instruction)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"📦 Dataset prepared:")
print(f"   Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")
print(f"   Sample:\n{dataset['train'][0]['text'][:200]}...")

---

## 3 · Training Loop with Live Tracking

### 🤔 Understanding Training Hyperparameters

| Parameter | What It Does | Too Low | Too High |
|-----------|-------------|---------|----------|
| `learning_rate` | How big each update step is | Model learns nothing | Model diverges (loss = NaN) |
| `epochs` | How many full passes over the data | Underfitting | Overfitting |
| `batch_size` | Samples per GPU per step | Noisy gradients | Out of memory |
| `gradient_accumulation` | Simulates larger batch | More noise | Slower training |
| `warmup_ratio` | Gradually increases LR at start | May diverge early | Wastes training time |

**🤔 Why use cosine learning rate schedule?**  
Cosine scheduling starts at the full learning rate, gradually decreases following a cosine curve, and reaches near-zero at the end. This gives the model strong learning early and gentle refinement later.

**🤔 What is gradient checkpointing?**  
Normally, training stores all intermediate computations in GPU memory (for backpropagation). Gradient checkpointing re-computes them instead of storing them, trading compute time for memory. This allows training larger models on smaller GPUs.

In [ ]:
# Cell 7 — Configure Hugging Face Trainer with W&B integration
#
# The TrainingArguments object is the CENTRAL CONFIGURATION for training.
# Every setting here affects training behavior, memory, speed, or quality.

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./lora-checkpoints",
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=0.01,                # L2 regularization to prevent overfitting
    warmup_ratio=0.03,                # warm up for 3% of training
    lr_scheduler_type="cosine",        # gradually decay learning rate
    logging_steps=10,                  # log to W&B every 10 steps
    save_strategy="epoch",             # save a checkpoint after each epoch
    evaluation_strategy="epoch",       # evaluate after each epoch
    bf16=True,                         # use bfloat16 for faster computation
    report_to="wandb",                 # send metrics to W&B automatically
    optim="paged_adamw_8bit",          # memory-efficient optimizer
    gradient_checkpointing=True,       # trade compute for memory
    max_grad_norm=0.3,                 # clip gradients to prevent exploding
    seed=42,                           # reproducibility
)

effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f"✅ Effective batch size: {effective_batch} (4 per GPU × 8 accumulation steps)")

In [ ]:
# Cell 8 — Initialize SFT Trainer and begin training
#
# SFTTrainer = Supervised Fine-Tuning Trainer
# It wraps the standard HuggingFace Trainer with features needed for LLM fine-tuning:
# - Automatic text tokenization
# - Sequence packing (pack multiple short examples into one sequence for efficiency)
# - Proper loss masking (only compute loss on the response, not the instruction)
#
# 🤔 What does packing=True do?
# Without packing: each batch item is one example, padded to max_seq_length
#   [Example1 + PAD PAD PAD PAD] [Example2 + PAD PAD PAD]
# With packing: multiple examples are concatenated into one sequence
#   [Example1 + Example2 + Example3] (no wasted padding)
# This can speed up training by 2-5x for datasets with short examples.

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=config.max_seq_length,
    tokenizer=tokenizer,
    args=training_args,
    packing=True,
)

print("🚀 Starting LoRA fine-tuning...")
train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Train loss: {train_result.training_loss:.4f}")
print(f"   Runtime: {train_result.metrics['train_runtime']:.0f}s")

In [ ]:
# Cell 9 — Evaluate and log generation quality
#
# Evaluation has TWO parts:
# 1. QUANTITATIVE: eval loss and perplexity (automatic metrics)
# 2. QUALITATIVE: generate actual text and inspect it (human review)
#
# 🤔 What is perplexity?
# Perplexity measures how "surprised" the model is by the test data.
# Lower = better. A perplexity of 10 means the model is as confused as if
# it had to choose between 10 equally likely options at each token.
#
# 🤔 Why do we need QUALITATIVE evaluation too?
# Because a model can have low perplexity but still generate nonsensical text.
# Always inspect actual outputs before deploying!

eval_results = trainer.evaluate()

test_prompts = [
    "### Instruction:\nExplain what a neural network is.\n\n### Response:\n",
    "### Instruction:\nWhat is overfitting?\n\n### Response:\n",
]

model.eval()
print("📊 Generation Samples:")
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7, do_sample=True, top_p=0.9)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(response[:300])

wandb.log({"eval_loss": eval_results["eval_loss"], "eval_perplexity": 2 ** eval_results["eval_loss"]})
print(f"\n📈 Eval loss: {eval_results['eval_loss']:.4f} | Perplexity: {2 ** eval_results['eval_loss']:.2f}")

---

## 4 · Registry Promotion & Artifact Management

### 🤔 Why Merge LoRA Weights?

During training, the LoRA adapters are SEPARATE from the base model. For deployment, you have two options:

| Option | Pros | Cons | When to Use |
|--------|------|------|-------------|
| **Keep separate** (adapter only) | Tiny file (~50MB), can swap adapters | Need base model + adapter at inference | Multiple adapters for different tasks |
| **Merge into base** | Single model file, simpler deployment | Larger file, can't swap adapters | Single task, production deployment |

### 🤔 What Are the Model Registry Stages?

```
Development → Staging → Production → Archived
    │              │          │            │
    │              │          │            └─ Old versions, kept for rollback
    │              │          └─ Serving live traffic
    │              └─ Being tested by QA/automated tests
    └─ Just trained, not yet tested
```

**Real-world scenario:**  
Your CI/CD pipeline (Notebook 04) automatically promotes a model from Development → Staging ONLY if it passes all behavioral tests. A human reviewer then approves Staging → Production.

In [ ]:
# Cell 10 — Merge LoRA weights and save final model
from peft import PeftModel

# Save adapter SEPARATELY first (for flexibility)
ADAPTER_PATH = "./lora-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"✅ LoRA adapter saved to: {ADAPTER_PATH}")
print(f"   Adapter size: ~{sum(f.stat().st_size for f in __import__('pathlib').Path(ADAPTER_PATH).rglob('*') if f.is_file()) / 1e6:.1f} MB")

# Merge adapters INTO base model for simpler deployment
merged_model = model.merge_and_unload()
MERGED_PATH = "./merged-model"
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print(f"✅ Merged model saved to: {MERGED_PATH}")

In [ ]:
# Cell 11 — Register model in MLflow
#
# WHAT IS HAPPENING?
# 1. Start an MLflow run (creates a tracked experiment entry)
# 2. Log all parameters (so we can reproduce this model)
# 3. Log metrics (so we can compare model versions)
# 4. Upload the adapter as an artifact (the actual model files)
# 5. Register the model (add it to the registry catalog)
#
# After this, the model has a NAME + VERSION in the registry.

import mlflow

with mlflow.start_run(run_name="lora-phi2-instruction-v1") as ml_run:
    mlflow.log_params({"base_model": MODEL_NAME, "lora_r": config.lora_r, "lora_alpha": config.lora_alpha, "quantization": config.quantization, "epochs": config.epochs, "learning_rate": config.learning_rate})
    mlflow.log_metrics({"train_loss": train_result.training_loss, "eval_loss": eval_results["eval_loss"], "eval_perplexity": 2 ** eval_results["eval_loss"], "trainable_params_pct": trainable_pct})
    mlflow.log_artifacts(ADAPTER_PATH, artifact_path="lora-adapter")
    registered_model = mlflow.register_model(f"runs:/{ml_run.info.run_id}/lora-adapter", "phi2-instruction-lora")

    print(f"✅ Model registered: phi2-instruction-lora v{registered_model.version}")

In [ ]:
# Cell 12 — Promote model to Staging
#
# This is the FIRST promotion step. The model goes from "just registered"
# to "Staging" — meaning it's ready for testing.
#
# 🤔 Why not go directly to Production?
# Because untested models in production = outages.
# The Staging stage is where behavioral tests (Notebook 04) run.
# Only after ALL tests pass does the model get promoted to Production.
#
# archive_existing_versions=True: if there's already a model in Staging,
# move it to Archived. This ensures only ONE version is in Staging at a time.

from mlflow.tracking import MlflowClient

client = MlflowClient()
client.transition_model_version_stage(
    name="phi2-instruction-lora",
    version=registered_model.version,
    stage="Staging",
    archive_existing_versions=True,
)
client.update_model_version(
    name="phi2-instruction-lora",
    version=registered_model.version,
    description=f"LoRA fine-tuned {MODEL_NAME}. Rank={config.lora_r}. Eval loss={eval_results['eval_loss']:.4f}.",
)

model_version = client.get_model_version("phi2-instruction-lora", registered_model.version)
print(f"✅ Model promoted to: {model_version.current_stage}")
print(f"   Name: {model_version.name} | Version: {model_version.version}")

wandb.finish()
print("🏁 Experiment complete")

---

## 5 · Knowledge Distillation: Compressing Large Models

### 🧠 What is Knowledge Distillation?

**Analogy:** A senior professor (teacher model, 70B params) teaches a junior lecturer (student model, 7B params). The student learns not just the right answers, but HOW the teacher thinks — which wrong answers are "almost right".

### The KD Loss Function

$$\mathcal{L}_{KD} = \alpha \mathcal{L}_{CE}(y, \hat{y}_{student}) + (1-\alpha) T^2 \mathcal{L}_{KL}\left( \sigma\left(\frac{z_{teacher}}{T}\right), \sigma\left(\frac{z_{student}}{T}\right) \right)$$

| Symbol | Meaning | Effect |
|--------|---------|--------|
| **T (temperature)** | Softens probability distributions | Higher T = softer, more knowledge transfer |
| **α (alpha)** | Balance hard vs soft targets | 0.5 = equal weight |
| **CE loss** | Standard classification loss (hard labels) | Learns correct answers |
| **KL divergence** | Diff between teacher/student distributions | Mimics teacher's reasoning |

### ⚖️ KD vs LoRA vs Full Fine-Tuning

| Method | Output | Cost | When to Use |
|--------|--------|------|-------------|
| **Knowledge Distillation** | Smaller, faster model | Medium | Deploy to edge/mobile |
| **LoRA** | Same-size model, new behavior | Low | Add task-specific skills |
| **Full fine-tuning** | Same-size model, deep changes | High | Maximum quality needed |

In [ ]:
# Cell 13 — Knowledge Distillation demo
#
# Train a small student network to mimic a larger teacher.
# Student learns from SOFT targets (teacher probabilities) + HARD targets (true labels).

import torch
import torch.nn as nn
import torch.nn.functional as F

class TeacherNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.fc(x)

class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x): return self.fc(x)

def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    """Combined KD loss: soft targets from teacher + hard targets from labels."""
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean') * (T * T)
    hard_loss = F.cross_entropy(student_logits, labels)
    return alpha * hard_loss + (1 - alpha) * soft_loss

teacher = TeacherNet()
student = StudentNet()
X = torch.randn(64, 784)
y = torch.randint(0, 10, (64,))

with torch.no_grad():
    teacher_out = teacher(X)
student_out = student(X)
loss = distillation_loss(student_out, teacher_out, y)

t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f"Teacher params: {t_params:,}")
print(f"Student params: {s_params:,}")
print(f"Compression:    {t_params / s_params:.1f}x smaller")
print(f"KD Loss:        {loss.item():.4f}")

---

## 6 · Magnitude-Based Weight Pruning

### 🧠 What is Pruning?

Pruning removes weights that contribute minimally to the output — like trimming dead branches.

### Pruning Methods (2026)

| Method | How | Sparsity | Quality Loss |
|--------|-----|---------|-------------|
| **Magnitude pruning** | Remove smallest weights | 30-50% | 0.5-2% |
| **Wanda** | Weight + activation aware | 50%+ | <1% |
| **SparseGPT** | One-shot LLM pruning | 50-60% | <1% |

### 🤔 When to Prune vs Quantize vs Distill?

| Goal | Best Technique |
|------|----------------|
| Reduce memory | Quantization (4-bit/8-bit) |
| Reduce compute (FLOPs) | Pruning (sparse operations) |
| Reduce model size entirely | Knowledge Distillation |
| Maximum compression | All three combined |

In [ ]:
# Cell 14 — Magnitude-based weight pruning
# Extract weights, compute threshold for 30% sparsity, apply mask, visualize

import matplotlib.pyplot as plt

layer = nn.Linear(128, 64)
weights = layer.weight.data.clone()

SPARSITY = 0.3  # Remove 30% of weights
threshold = torch.quantile(weights.abs(), SPARSITY)
mask = (weights.abs() >= threshold).float()
pruned_weights = weights * mask

actual_sparsity = 1.0 - mask.mean().item()
print(f"Pruning Results:")
print(f"  Target sparsity: {SPARSITY:.0%}")
print(f"  Actual sparsity: {actual_sparsity:.1%}")
print(f"  Weights before:  {(weights != 0).sum().item()}")
print(f"  Weights after:   {(pruned_weights != 0).sum().item()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(weights.numpy(), cmap="RdBu", aspect="auto")
axes[0].set_title("Original Weights", fontweight="bold")
axes[1].imshow(pruned_weights.numpy(), cmap="RdBu", aspect="auto")
axes[1].set_title(f"After {SPARSITY:.0%} Pruning", fontweight="bold")
plt.tight_layout()
plt.savefig("pruning_pattern.png", dpi=150)
plt.show()

---

## 7 · Automated Hyperparameter Tuning with Optuna + MLflow

### 🤔 Why Automate Hyperparameter Search?

Manually trying values is slow. Optuna uses **Bayesian optimization** to intelligently explore.

| Method | Trials Needed | Quality | Cost |
|--------|:---:|:---:|:---:|
| **Manual** | 5-10 | Low | Low |
| **Grid search** | 100+ | Medium | High |
| **Random search** | 20-50 | Good | Medium |
| **Bayesian (Optuna)** | 10-30 | Best | Low-Medium |

In [ ]:
# Cell 15 — Optuna hyperparameter search with MLflow logging
!pip install -q optuna
import optuna
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score

X_opt, y_opt = make_classification(n_samples=1000, n_features=20, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }
    model = GradientBoostingClassifier(**params, random_state=42)
    score = cross_val_score(model, X_opt, y_opt, cv=3, scoring="accuracy").mean()
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("cv_accuracy", score)
    return score

mlflow.set_experiment("optuna-hpo-search")
with mlflow.start_run(run_name="optuna-parent"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=15, show_progress_bar=True)

print(f"Best accuracy: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

---
## Knowledge Check

### Question 1: LoRA Rank
You set `lora_r=4` and the model barely changes after fine-tuning. What went wrong and how would you fix it?

<details>
<summary>Click to reveal answer</summary>

The rank is too low -- the adapters don't have enough capacity to learn the task. Increase `lora_r` to 16 or 32 (higher rank = more learning capacity). Also ensure `lora_alpha` is proportional (typically 2x the rank). If r=4 alpha=8 didn't work, try r=16 alpha=32.
</details>

### Question 2: QLoRA Memory
A 7B parameter model at FP16 requires ~14GB VRAM. With QLoRA (4-bit), approximately how much VRAM is needed? Why?

<details>
<summary>Click to reveal answer</summary>

~4-5GB VRAM. 4-bit quantization compresses each parameter from 16 bits to 4 bits (4x reduction: 14GB / 3.5 = ~4GB). The extra ~1GB is overhead for the LoRA adapter weights (which train in FP16), optimizer states, and activation memory.
</details>

### Question 3: Model Registry Stages
A new model version is trained. Describe the correct lifecycle path and what happens at each stage.

<details>
<summary>Click to reveal answer</summary>

Development (just trained, metrics logged) -> Staging (runs behavioral tests, invariance tests, load tests) -> Production (serves live traffic, only after all gates pass) -> Archived (when a newer version replaces it). At each stage, the model is registered with its version, metrics, and who approved the transition.
</details>

### Question 4: Perplexity
Model A has perplexity 15. Model B has perplexity 150. Which is better and what does this mean?

<details>
<summary>Click to reveal answer</summary>

Model A (perplexity 15) is **much** better. Perplexity measures how "confused" the model is: lower = better. A perplexity of 15 means the model acts as if choosing between 15 equally likely options at each token. 150 means it's extremely uncertain -- essentially guessing randomly among 150 options.
</details>


---
## Practical Practice

### Exercise 1: LoRA Configuration
You need to fine-tune a 3B parameter model on a GPU with 8GB VRAM for a customer support chatbot. Configure the LoraConfig and BitsAndBytesConfig with appropriate settings. Explain each choice.

### Exercise 2: Experiment Comparison
You ran 3 experiments with different LoRA ranks. Write the W&B code to log configs and compare them.

### Exercise 3: Knowledge Distillation Loss
Given a teacher output `[0.1, 0.7, 0.2]` and student output `[0.3, 0.4, 0.3]` for a 3-class problem with true label `1`, compute the KD loss with T=2, alpha=0.5.


In [ ]:
# ===== EXERCISE SOLUTIONS =====
import torch
import torch.nn.functional as F

print('Ex 1: QLoRA Config for 3B model on 8GB GPU')
print('''BitsAndBytesConfig(
    load_in_4bit=True,           # Must use 4-bit for 8GB GPU
    bnb_4bit_quant_type="nf4",    # NF4 is best for neural network weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in bf16 for stability
    bnb_4bit_use_double_quant=True,  # Extra memory savings
)
LoraConfig(
    r=16,                  # Good starting rank
    lora_alpha=32,         # 2x rank is the rule of thumb
    target_modules=["q_proj", "k_proj", "v_proj"],  # Attention layers
    lora_dropout=0.05,     # Small dropout to prevent overfitting
    task_type="CAUSAL_LM",
)''')

print('\nEx 2: W&B Experiment Comparison')
print('''for rank in [8, 16, 32]:
    run = wandb.init(project="lora-comparison", config={"lora_r": rank, "lora_alpha": rank*2})
    # ... train ...
    wandb.log({"eval_loss": eval_loss, "perplexity": 2**eval_loss})
    wandb.finish()
# Then in W&B UI: compare runs side-by-side''')

print('\nEx 3: KD Loss computation')
teacher_logits = torch.tensor([[0.1, 0.7, 0.2]])
student_logits = torch.tensor([[0.3, 0.4, 0.3]])
labels = torch.tensor([1])
T = 2.0
alpha = 0.5

soft_loss = F.kl_div(
    F.log_softmax(student_logits / T, dim=1),
    F.softmax(teacher_logits / T, dim=1),
    reduction='batchmean') * (T * T)
hard_loss = F.cross_entropy(student_logits, labels)
kd_loss = alpha * hard_loss + (1 - alpha) * soft_loss
print(f'  Soft loss (KL div): {soft_loss.item():.4f}')
print(f'  Hard loss (CE):     {hard_loss.item():.4f}')
print(f'  KD loss (combined): {kd_loss.item():.4f}')


---

## 🎯 Summary & Key Takeaways

| Step | Action | Tool | Why |
|------|--------|------|-----|
| 1 | Experiment tracking with full config | W&B + MLflow | Reproduce any experiment |
| 2 | 4-bit QLoRA adapter injection | BitsAndBytes + PEFT | Train on consumer GPUs |
| 3 | SFT training with live loss streaming | HF Trainer + W&B | Monitor training in real-time |
| 4 | Adapter merge, registry upload, Staging promotion | MLflow | Lifecycle management |
| 5 | Knowledge Distillation | PyTorch | Compress large teacher into small student |
| 6 | Magnitude-based weight pruning | PyTorch | Remove unnecessary weights for efficiency |
| 7 | Automated hyperparameter tuning | Optuna + MLflow | Find optimal settings automatically |

### 🧠 Key Questions

1. **"Should I fine-tune or use prompting?"** → Fine-tune when you need consistent behavior, lower latency, or domain-specific knowledge. Use prompting for general tasks.
2. **"How much data do I need for fine-tuning?"** → LoRA: 1K-10K examples is often enough. Full fine-tuning: 10K-100K+.
3. **"How do I know if fine-tuning worked?"** → Lower eval loss, lower perplexity, AND qualitative inspection of generated outputs.
4. **"When should I use KD vs pruning vs quantization?"** → KD for smaller architecture, pruning for fewer FLOPs, quantization for lower memory. Combine all three for maximum efficiency.

**Next -->** `05_cicd_automation.ipynb` -- CI/CD & Pipeline Automation
